# Genetic Algorithm

Changed on Thu Dec 04 12:53:40 2025

@author: gustaaragao

## Objetivo
Executar e reportar resultados do **Algoritmo Genético (Alg. 20)** no problema **Ackley** (multimodal), conforme as seções 3.1–3.2 do *Essentials of Metaheuristics*, usando:
- **Seleção Fitness-Proportionate** (Alg. 30)
- **Cruzamento Two-Point** (Alg. 24)
- **Mutação Bounded Uniform Convolution** (Alg. 8)

O notebook gera:
- Tabela com **média ± desvio**, **mediana**, **melhor** e **pior** (em múltiplas seeds)
- **Um gráfico** de qualidade: **curva de convergência** (média ± desvio) do melhor fitness ao longo das avaliações

In [1]:
# Imports e setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.problems.Ackley import Ackley
from src.Solution import Solution
from src.MOEAs.crossovers.TwoPointCrossover import TwoPointCrossover
from src.MOEAs.mutations.BoundedUniformConvolutionMutation import BoundedUniformConvolutionMutation
from src.FitnessProportionate import FitnessProportionate

def make_ackley(n_vars=5, a=20.0, b=0.2, c=2*np.pi):
    return Ackley(numberOfDecisionVariables=n_vars, a=a, b=b, c=c)

def random_solution(problem: Ackley) -> Solution:
    """Gera uma solução aleatória dentro dos limites do problema.

    Nota: fazemos isso aqui no notebook para evitar depender da implementação
    genérica de `Problem.generateSolution()` (framework).
    """
    lower, upper = problem.decisionVariablesLimit
    s = Solution(problem.numberOfObjectives, problem.numberOfDecisionVariables)
    s.decisionVariables = [
        float(np.random.uniform(lower[i], upper[i]))
        for i in range(problem.numberOfDecisionVariables)
    ]
    return s

## Problema Ackley (visão geral)
A função de Ackley (para $D$ variáveis) pode ser escrita como:

$$
f(x) = -a\,\exp\left(-b\,\sqrt{\frac{1}{D}\sum_{i=1}^{D} x_i^2}\right) - \exp\left(\frac{1}{D}\sum_{i=1}^{D} \cos(c\,x_i)\right) + a + e
$$

- **Mínimo global**: $x^* = [0,\dots,0]$ e $f(x^*) = 0$
- **Multimodalidade**: muitos mínimos locais, o que dificulta métodos puramente locais (ex.: Hill Climbing)

## Experimento principal — GA no Ackley (D=30)
Este experimento implementa e executa o **Algoritmo Genético (Alg. 20)** no problema **Ackley** com a configuração descrita no relatório:
- $D=30$
- população = 100
- $p_{cross}=0.8$ (Two-Point Crossover, Alg. 24)
- $p_{mut}=0.1$ (Bounded Uniform Convolution, Alg. 8)
- seleção Fitness-Proportionate (Alg. 30)
- parada em **50.000 avaliações**

> Observação: para produzir **média ± desvio**, o notebook roda múltiplas seeds e calcula estatísticas.

In [2]:
# Parâmetros do experimento (como no texto do relatório)
D = 30
popsize = 100
p_cross = 0.8
p_mut = 0.1
range_noise = 1.0  # ajuste se você definiu outro r para a mutação
max_evals = 50_000

# Número de repetições (seeds) para estatísticas
n_runs = 20
seeds = list(range(n_runs))

assert popsize % 2 == 0, "populationSize deve ser par no Alg. 20"
print("Config:", {
    "D": D,
    "popsize": popsize,
    "p_cross": p_cross,
    "p_mut": p_mut,
    "range_noise": range_noise,
    "max_evals": max_evals,
    "n_runs": n_runs,
})

Config: {'D': 30, 'popsize': 100, 'p_cross': 0.8, 'p_mut': 0.1, 'range_noise': 1.0, 'max_evals': 50000, 'n_runs': 20}


In [3]:
def run_ga_with_trace(seed: int):
    """Executa um GA (Alg. 20) e guarda a curva de convergência.

    Retorna:
    - best_final (float)
    - evals_hist (np.ndarray)  # número de avaliações ao longo do tempo
    - best_hist (np.ndarray)   # melhor objetivo até cada ponto
    """
    np.random.seed(seed)
    problem = make_ackley(n_vars=D)
    lower, upper = problem.decisionVariablesLimit

    selection = FitnessProportionate()
    crossover = TwoPointCrossover(distributionIndex=None, crossoverProbability=p_cross)
    mutation = BoundedUniformConvolutionMutation(mutationProbability=p_mut, range_noise=range_noise)

    # População inicial (não avaliada)
    population = [random_solution(problem) for _ in range(popsize)]

    evaluations = 0
    best_so_far = float("inf")

    # Avalia população inicial
    for ind in population:
        problem.evaluate(ind)
        evaluations += 1
        if ind.objectives[0] < best_so_far:
            best_so_far = float(ind.objectives[0])

    evals_hist = [evaluations]
    best_hist = [best_so_far]

    # Loop principal (gerações)
    while evaluations < max_evals:
        offspring = []
        for _ in range(popsize // 2):
            parent_a = selection.select(population).clone()
            parent_b = selection.select(population).clone()

            children = crossover.crossover([parent_a, parent_b], lower, upper)
            children[0] = mutation.mutate(children[0], lower, upper)
            children[1] = mutation.mutate(children[1], lower, upper)
            offspring.extend(children)

        population = offspring

        # Avalia descendentes
        for ind in population:
            problem.evaluate(ind)
            evaluations += 1
            if ind.objectives[0] < best_so_far:
                best_so_far = float(ind.objectives[0])
        evals_hist.append(evaluations)
        best_hist.append(best_so_far)

    return best_so_far, np.asarray(evals_hist), np.asarray(best_hist)

In [4]:
# Executa N repetições e consolida estatísticas + curva média
results = []
histories = []
evals_ref = None

for seed in seeds:
    best_final, evals_hist, best_hist = run_ga_with_trace(seed)
    results.append({"seed": seed, "best_final": best_final})
    histories.append(best_hist)
    if evals_ref is None:
        evals_ref = evals_hist
    else:
        # Segurança: garante que todas as curvas têm o mesmo eixo-x
        if not np.array_equal(evals_ref, evals_hist):
            raise ValueError("Eixos de avaliações diferentes entre execuções")

df = pd.DataFrame(results)
df

,seed,best_final
0,0,20.486694
1,1,20.465662
2,2,20.504738
3,3,20.419307
4,4,20.432927
5,5,20.455419
6,6,20.491032
7,7,20.409734
8,8,20.532126
9,9,20.445744
